# Phase 3: YOLO Detector Evaluation

This notebook evaluates the YOLO model trained in Phase 2 on the unseen **test set**.
It computes performance metrics, generates PR curves and confusion matrices, and performs a confidence sweep.

In [1]:
# 1. Setup and Google Drive Mount
!pip install ultralytics pandas matplotlib pyyaml > /dev/null 2>&1
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

PROJECT_ROOT = '/content/drive/MyDrive/sem_defect_project'
DATA_YAML = f"{PROJECT_ROOT}/dataset_yolo_single_class/data.yaml"

# Dynamically identify which experiment name was used during training
BEST_PT_V12 = f"{PROJECT_ROOT}/runs/detect/EXP-02-YOLOv12-SingleClass-Baseline/weights/best.pt"
BEST_PT_V11 = f"{PROJECT_ROOT}/runs/detect/EXP-02-YOLO11-SingleClass-Baseline/weights/best.pt"

EVAL_DIR = f"{PROJECT_ROOT}/runs/detect/EVALUATION"

if os.path.exists('/content/drive/MyDrive'):
    print("✅ Google Drive is mounted.")
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✅ Google Drive successfully mounted.")
    except Exception as e:
        raise RuntimeError("Please mount Drive using the Colab web UI.") from e

if os.path.exists(BEST_PT_V12):
    BEST_PT = BEST_PT_V12
elif os.path.exists(BEST_PT_V11):
    BEST_PT = BEST_PT_V11
else:
    raise FileNotFoundError("Trained model not found! Did Phase 2 complete successfully?")

print(f"✅ Using trained model checkpoint: {BEST_PT}")
os.makedirs(EVAL_DIR, exist_ok=True)
print("✅ Setup complete.")

✅ Google Drive is mounted.
✅ Using trained model checkpoint: /content/drive/MyDrive/sem_defect_project/runs/detect/EXP-02-YOLO11-SingleClass-Baseline/weights/best.pt
✅ Setup complete.


In [2]:
# 2. Validation Set Evaluation
print("--- Evaluating on VALIDATION set ---")
model = YOLO(BEST_PT)

val_metrics = model.val(
    data=DATA_YAML,
    split='val',
    project=EVAL_DIR,
    name="val_baseline",
    plots=True
)

print(f"\nValidation mAP@50: {val_metrics.box.map50}")
print(f"Validation mAP@50-95: {val_metrics.box.map}")

--- Evaluating on VALIDATION set ---
Ultralytics 8.4.150 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.1 ms, read: 33.3±16.7 MB/s, size: 49.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/sem_defect_project/dataset_yolo_single_class/valid/labels.cache... 31 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 31/31 6.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.2it/s 0.6s0.9s
                   all         31        184      0.582      0.614      0.585      0.354
Speed: 2.5ms preprocess, 6.4ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /content/drive/MyDrive/sem_defect_project/runs/detect/EVALUATION

In [3]:
# 3. Untouched TEST Set Evaluation
print("\n--- Evaluating on TEST set ---")
test_metrics = model.val(
    data=DATA_YAML,
    split='test',
    project=EVAL_DIR,
    name="test_baseline",
    plots=True,
    save_json=True
)

print(f"\nTest mAP@50: {test_metrics.box.map50}")
print(f"Test mAP@50-95: {test_metrics.box.map}")

# Extract Precision and Recall
test_p = test_metrics.box.mp
test_r = test_metrics.box.mr
test_f1 = 2 * (test_p * test_r) / (test_p + test_r + 1e-6)

print(f"Test Precision: {test_p:.4f}")
print(f"Test Recall: {test_r:.4f}")
print(f"Test F1-Score: {test_f1:.4f}")


--- Evaluating on TEST set ---
Ultralytics 8.4.150 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.1 ms, read: 35.1±18.6 MB/s, size: 50.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/sem_defect_project/dataset_yolo_single_class/test/labels... 30 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30/30 154.7it/s 0.2s.2s
val: New cache created: /content/drive/MyDrive/sem_defect_project/dataset_yolo_single_class/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.9it/s 1.0s1.3s
                   all         30        122       0.76       0.59      0.688      0.456
Speed: 3.7ms preprocess, 7.8ms inference, 0.0ms loss, 7.6ms p

In [4]:
# 4. Confidence Threshold Analysis
print("\n--- Running Confidence Threshold Sweep on TEST set ---")
thresholds = [0.15, 0.25, 0.40, 0.50, 0.60]
sweep_results = []

for conf in thresholds:
    print(f"\nEvaluating at conf={conf}...")
    res = model.val(
        data=DATA_YAML,
        split='test',
        conf=conf,
        project=EVAL_DIR,
        name=f"test_conf_{conf}",
        plots=False,
        verbose=False
    )
    
    p = res.box.mp
    r = res.box.mr
    f1 = 2 * (p * r) / (p + r + 1e-6)
    sweep_results.append({
        'conf': conf,
        'precision': float(p),
        'recall': float(r),
        'f1': float(f1),
        'mAP50': float(res.box.map50)
    })

df_sweep = pd.DataFrame(sweep_results)
print("\nThreshold Sweep Results:")
print(df_sweep.to_string(index=False))


--- Running Confidence Threshold Sweep on TEST set ---

Evaluating at conf=0.15...
Ultralytics 8.4.150 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.1 ms, read: 37.8±17.8 MB/s, size: 56.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/sem_defect_project/dataset_yolo_single_class/test/labels.cache... 30 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30/30 9.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 4.5it/s 0.4s1.1s
                   all         30        122       0.76       0.59      0.582      0.408
Speed: 0.7ms preprocess, 3.6ms inference, 0.0ms loss, 1.1ms postprocess per image

Evaluating at conf=0.25...
Ultraly

In [5]:
# 5. Generate Predictions for Visual Analysis (Error Analysis directories)
import shutil

print("\n--- Running Inference & Error Analysis Dirs ---")
test_images_dir = f"{PROJECT_ROOT}/dataset_yolo_single_class/test/images"

# Run standard inference to generate bounding box predictions on test images
preds = model.predict(
    source=test_images_dir,
    conf=0.25, # good balance from sweep
    project=EVAL_DIR,
    name="test_predictions",
    save=True,
    save_txt=True,
    save_conf=True
)
print(f"Predictions saved visually to {EVAL_DIR}/test_predictions")


--- Running Inference & Error Analysis Dirs ---

image 1/30 /content/drive/MyDrive/sem_defect_project/dataset_yolo_single_class/test/images/3D-Dandelion0004j_jpg.rf.c27c8cd565a0c5c50aeb461c9b0ab848.jpg: 512x512 3 Defects, 9.1ms
image 2/30 /content/drive/MyDrive/sem_defect_project/dataset_yolo_single_class/test/images/cnc14_march220006h_jpg.rf.91ac60dc97be5d17660b707311284672.jpg: 512x512 2 Defects, 20.2ms
image 3/30 /content/drive/MyDrive/sem_defect_project/dataset_yolo_single_class/test/images/nav10001a3_jpg.rf.306ea652cb173d06f70c4e7f66827123.jpg: 512x512 3 Defects, 21.4ms
image 4/30 /content/drive/MyDrive/sem_defect_project/dataset_yolo_single_class/test/images/nav10006aa1_jpg.rf.c7c7b108654b3bd9ea5584bd54716ea0.jpg: 512x512 5 Defects, 9.0ms
image 5/30 /content/drive/MyDrive/sem_defect_project/dataset_yolo_single_class/test/images/nav20004a5_jpg.rf.eebfb0c0353a684dd2f53ec3bdab2ce8.jpg: 512x512 7 Defects, 9.6ms
image 6/30 /content/drive/MyDrive/sem_defect_project/dataset_yolo_single

In [6]:
# 6. Export Final Summary JSON
summary = {
    'val': {
        'precision': float(val_metrics.box.mp),
        'recall': float(val_metrics.box.mr),
        'mAP50': float(val_metrics.box.map50),
        'mAP50-95': float(val_metrics.box.map)
    },
    'test': {
        'precision': float(test_p),
        'recall': float(test_r),
        'f1': float(test_f1),
        'mAP50': float(test_metrics.box.map50),
        'mAP50-95': float(test_metrics.box.map)
    },
    'sweep': sweep_results
}

summary_path = f"{EVAL_DIR}/eval_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=4)

print(f"\n✅ Evaluation completed. Summary saved to {summary_path}")
print("\n================ FINAL RESULTS ================")
print(json.dumps(summary, indent=4))
print("\nPlease copy these results back or review them locally to proceed with Phase 3 Report generation.")


✅ Evaluation completed. Summary saved to /content/drive/MyDrive/sem_defect_project/runs/detect/EVALUATION/eval_summary.json

================ FINAL RESULTS ================
{
    "val": {
        "precision": 0.5815296765398069,
        "recall": 0.6141304347826086,
        "mAP50": 0.5853457764862716,
        "mAP50-95": 0.35390805985697754
    },
    "test": {
        "precision": 0.7597502398848843,
        "recall": 0.5901639344262295,
        "f1": 0.6643042456064694,
        "mAP50": 0.6876475040842585,
        "mAP50-95": 0.45553437661204577
    },
    "sweep": [
        {
            "conf": 0.15,
            "precision": 0.7597502398848843,
            "recall": 0.5901639344262295,
            "f1": 0.6643042456064694,
            "mAP50": 0.5820740539874873
        },
        {
            "conf": 0.25,
            "precision": 0.7635299398620958,
            "recall": 0.5901639344262295,
            "f1": 0.6657450536600928,
            "mAP50": 0.5542081166215499
        }